# Traveller Sector Generator

Generates a whole sector and renders it in the style of the Classic Traveller
maps, using the `worldmaker` package.

This notebook is a thin front-end over the tested package; the rules are not
duplicated here.


In [ ]:
import random

import worldmaker as wm

random.seed(1105)

# A full sector is 32x40 hexes: sixteen 8x10 subsectors, generated in one pass
# so polities, routes and names are coherent across the whole map.
sector = wm.generate_full_sector("Foreven Reach")
print(f"{len(sector.systems)} systems in {sector.width}x{sector.height} hexes")
print(f"{len(sector.polities)} polities, "
      f"{sum(1 for r in sector.routes if r[2] == 'xboat')} Xboat links")

## Polities

In [ ]:
for p in sector.polities:
    print(f"{p.allegiance_code}  {p.name:34s} capital {p.capital_hex}  "
          f"{len(p.controlled_systems):3d} worlds  DI {p.defense_index:2d}  "
          f"{p.polity_type}")

unaligned = sum(1 for s in sector.systems.values() if s.allegiance == "Na")
print(f"\n{unaligned} independent systems")

## Subsectors

In [ ]:
for letter, name in sorted(sector.subsector_names.items()):
    idx = ord(letter) - 65
    col0, row0 = (idx % 4) * 8, (idx // 4) * 10
    count = sum(
        1 for h in sector.systems
        if col0 < int(h[:2]) <= col0 + 8 and row0 < int(h[2:]) <= row0 + 10)
    print(f"{letter}  {name:16s} {count:3d} systems")

## Sector map\n\nThe full foldout, in the classic black-ink style.

In [ ]:
from IPython.display import SVG, display

display(SVG(wm.generate_sector_svg(sector)))

## Subsector map\n\nA single subsector at Supplement-page scale.

In [ ]:
display(SVG(wm.generate_subsector_svg(sector, sector.subsector_names.get('A', 'Subsector A'))))

### Any other subsector

Pass the origin hex of the subsector window: subsector F starts at 0911.

In [ ]:
display(SVG(wm.generate_subsector_svg(
    sector, sector.subsector_names.get('F', 'Subsector F'),
    origin_col=9, origin_row=11)))

## Sector data file\n\nTravellermap-style `.sec` output.

In [ ]:
sec_data = wm.export_sector_sec_file(sector)
print("\n".join(sec_data.splitlines()[:25]))
print("...")

In [ ]:
# Write the maps and data to disk
with open("sector_map.svg", "w") as f:
    f.write(wm.generate_sector_svg(sector))
with open("sector.sec", "w") as f:
    f.write(sec_data)
print("written: sector_map.svg, sector.sec")